<h1 id="Aspects-(plus)-avanc&eacute;s-pour-la-bioinformatique:-recherche-dans-une-base-de-donn&eacute;es-distante,-version-JSON-et-MyGene.info">Aspects (plus) avanc&eacute;s pour la bioinformatique: recherche dans une base de donn&eacute;es distante, version XML et NCBI Entrez</h1>
<p>Le processus de collecte d'informations pour un processus d'annotation requiert des recherches sur des bases de donn&eacute;es pr&eacute;-existantes. De mani&egrave;re non-programmatique, on commence par aller sur une page web, on inscrit un ou plusieurs crit&egrave;res de recherche et le site nous retourne un ou des pages de r&eacute;sultats. Ok, c'est cool mais comment int&eacute;grer &ccedil;a dans une structure de donn&eacute;es? Il est tout &agrave; fait possible de le faire programmatiquement via Python. Dans les exemples qui suivent, nous allons utiliser le service MyGene.info pour faire notre recherche et notre collecte d'informations.</p>
<p>Le NCBI (<em>National Center for Biotechnology Information</em>) est le <em>uber</em> site de stockage des donn&eacute;es biologiques et chimiques (PubChem); le NCBI offre gratuitement un oc&eacute;an d'informations qu'on peut consulter pour cr&eacute;er des annotations selon nos besoins, accessibles via la suite d'outils EUtils dans un URL,</p>
<p>Dans l'exemple qui suit, on recherchera un ensemble de variations g&eacute;n&eacute;tiques en utilisant la base de donn&eacute;es dbVar du NCBI pour notre ami, le g&egrave;ne CTTN.&nbsp;La logique de recherche des infos reste la m&ecirc;me:</p>
<ul>
<li>On recherche un ensemble de donn&eacute;es (ici, les variations g&eacute;n&eacute;tiques) se retrouvant associ&eacute;es au g&egrave;ne CTTN en utilisant l'utilisaire ESearch;</li>
<li>Pour chaque variation retrouv&eacute;e, on veux avoir les informations disponibles sur cette variation en utilisant l'utilitaire EFetch.</li>
</ul>
<p>Pour transformer "instantan&eacute;ment" un flux xml en dictionnaire, il nous fait le module xmltodict, qu'on installe si n&eacute;cessaire avec pip</p>

In [8]:
!pip install xmltodict

zsh:1: /usr/local/bin/pip: bad interpreter: /System/Library/Frameworks/Python.framework/Versions/2.7/Resources/Python.app/Contents/MacOS/Python: no such file or directory
Defaulting to user installation because normal site-packages is not writeable


In [17]:
import requests
import xmltodict
# Pas necassaire pour le travail à faire mais
# simplement pour afficher ce qui revient du serveur
import xml.dom.minidom

# Preparation de la requete
headers = {'content-type': 'application/x-www-form-urlencoded'}
# Je limite à un seul résultat pour faire l'exemple :-)
search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=CTTN[Gene]&retMax=2&rettype=xml"
# Envoyez la requete
res = requests.get(search_url,headers=headers)
#
# Retirez les commentaires pour voir..
# Juste pour vous montrer de quoi à l'air le format XML :-)
#
#print("=========== Sortie XML ===========\n")
#temp = xml.dom.minidom.parseString(res.content) 
#new_xml = temp.toprettyxml(indent="  ") 
#print(new_xml) 
#print("==================================\n")

# C'est avec ça que nous allons travailler!
aDict = xmltodict.parse(res.content)
print(aDict)
# On recupère le contenu de la clé eSearchResult
# créé par le parsing du contenu XML
aDict = aDict["eSearchResult"]

#
# Maintenant on récupère les infos de chacun des éléments Id sous IdList
#
idList = aDict["IdList"]
idList = idList["Id"]
for i in idList:
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=snp&id="+i+"&retmode=xml"
    res = requests.get(fetch_url,headers=headers)
    aVar = xmltodict.parse(res.content)
    print(aVar)

{'eSearchResult': {'Count': '17007', 'RetMax': '2', 'RetStart': '0', 'IdList': {'Id': ['2135607633', '2135607342']}, 'TranslationSet': None, 'TranslationStack': {'TermSet': {'Term': 'CTTN[Gene]', 'Field': 'Gene', 'Count': '17007', 'Explode': 'N'}, 'OP': 'GROUP'}, 'QueryTranslation': 'CTTN[Gene]'}}
{'ExchangeSet': {'@xmlns:xsi': 'https://www.w3.org/2001/XMLSchema-instance', '@xmlns': 'https://www.ncbi.nlm.nih.gov/SNP/docsum', '@xsi:schemaLocation': 'https://www.ncbi.nlm.nih.gov/SNP/docsum ftp://ftp.ncbi.nlm.nih.gov/snp/specs/docsum_eutils.xsd', 'DocumentSummary': {'@uid': '2135607633', 'SNP_ID': '2135607633', 'ALLELE_ORIGIN': None, 'GLOBAL_MAFS': None, 'GLOBAL_POPULATION': None, 'GLOBAL_SAMPLESIZE': '0', 'SUSPECTED': None, 'CLINICAL_SIGNIFICANCE': None, 'GENES': {'GENE_E': {'NAME': 'CTTN', 'GENE_ID': '2017'}}, 'ACC': 'NC_000011.10', 'CHR': '11', 'HANDLE': 'EVA', 'SPDI': 'NC_000011.10:70436929:T:G', 'FXN_CLASS': '500B_downstream_variant,downstream_transcript_variant', 'VALIDATED': None, 